## Couverture conforme sous changement de distribution

Ce notebook exécute l'étude préenregistrée dans `docs/research_conformal_shift.md`. La question est de savoir quelles méthodes de prédiction conforme maintiennent la couverture nominale à la fois en moyenne et dans chaque pays, lorsque le modèle prédit des régions nouvelles ou des années futures.

**Déroulement**

- Le plan a été versionné avant tout calcul.
- Les cinq méthodes M0 à M4 sont comparées sur un banc d'essai interne au développement.
- La règle de décision du plan a désigné une méthode par jeu, figée et versionnée avant l'évaluation sur les ensembles de test.
- Toutes les méthodes sont ensuite rapportées sur les ensembles de test, évalués une seule fois.

**Préalable**

Le notebook suppose les expériences de `10_modeling.ipynb` et l'évaluation de `11_evaluation.ipynb` calculées. Il s'exécute en trois minutes environ.

In [1]:
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE))

from src.recherche.couverture import (
    CONDITIONS, CRITERE_REMPLI, METHODES, METHODES_RETENUES, decider, ecarts_au_m0, evaluation_finale, evaluer_banc,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)
JEUX = ["regional", "survey"]
COLONNES = ["condition", "methode", "niveau", "lignes", "couverture", "pire_pays", "pire_pays_nom",
            "ecart_moyen_pays", "largeur_mediane"]

### 1. Tests unitaires

Les tests de l'étude vérifient le quantile conforme, l'ordre hiérarchique des méthodes M2 et M3, les métriques par pays, la règle de décision et la répartition stable des groupes en deux moitiés.

In [2]:
execution = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-t", "."],
    cwd=RACINE, capture_output=True, text=True,
)
print(execution.stderr.strip().splitlines()[-3:])

['Ran 106 tests in 7.752s', '', 'OK']


### 2. Banc d'essai interne

La calibration utilise les erreurs de validation croisée des lignes anciennes. Les trois conditions évaluent des régions nouvelles des années anciennes, des années récentes de régions connues et des années récentes de régions nouvelles. La couverture du pire pays porte sur les pays d'au moins 200 lignes évaluées.

In [3]:
resultats, details, bancs = {}, {}, {}
for jeu in JEUX:
    resultats[jeu], details[jeu], bancs[jeu] = evaluer_banc(jeu)
    print(jeu, ": lignes par rôle", bancs[jeu]["role"].value_counts().to_dict())
    display(resultats[jeu][COLONNES].round(3))
    choix, critere = decider(resultats[jeu])
    print("  règle de décision :", choix, "| critère de couverture marginale rempli :", critere,
          "| choix figé dans le code :", METHODES_RETENUES[jeu], CRITERE_REMPLI[jeu])

regional : lignes par rôle {'calibration': 78704, 'spatiale': 31782, 'temporelle': 24371, 'spatio_temporelle': 24371}


,condition,methode,niveau,lignes,couverture,pire_pays,pire_pays_nom,ecart_moyen_pays,largeur_mediane
0,spatiale,M0,0.8,31782,0.805,0.695,Mali,0.062,2.956
1,spatiale,M0,0.9,31782,0.901,0.798,Mali,0.035,4.886
2,spatiale,M1,0.8,31782,0.802,0.696,South Africa,0.064,3.092
3,spatiale,M1,0.9,31782,0.902,0.794,Mali,0.037,5.100
4,spatiale,M2,0.8,31782,0.791,0.649,South Africa,0.044,2.868
5,spatiale,M2,0.9,31782,0.892,0.804,Mali,0.026,4.628
6,spatiale,M3,0.8,31782,0.782,0.649,South Africa,0.048,2.868
7,spatiale,M3,0.9,31782,0.883,0.804,Mali,0.028,4.735
8,spatiale,M4,0.8,31782,0.783,0.602,Mauritania,0.054,2.824
9,spatiale,M4,0.9,31782,0.890,0.737,Mauritania,0.034,4.252


  règle de décision : M0 | critère de couverture marginale rempli : False | choix figé dans le code : M0 False


survey : lignes par rôle {'calibration': 63089, 'spatiale': 27027, 'temporelle': 19204, 'spatio_temporelle': 19204}


,condition,methode,niveau,lignes,couverture,pire_pays,pire_pays_nom,ecart_moyen_pays,largeur_mediane
0,spatiale,M0,0.8,27027,0.793,0.712,Nigeria,0.038,9.836
1,spatiale,M0,0.9,27027,0.898,0.836,Guinea-Bissau,0.020,18.174
2,spatiale,M1,0.8,27027,0.793,0.714,Nigeria,0.050,10.061
3,spatiale,M1,0.9,27027,0.897,0.827,Guinea-Bissau,0.030,19.777
4,spatiale,M2,0.8,27027,0.795,0.770,Niger,0.016,10.041
5,spatiale,M2,0.9,27027,0.898,0.876,Togo,0.010,20.080
6,spatiale,M3,0.8,27027,0.793,0.743,Niger,0.016,10.253
7,spatiale,M3,0.9,27027,0.898,0.870,Niger,0.010,18.736
8,spatiale,M4,0.8,27027,0.791,0.739,Togo,0.027,9.473
9,spatiale,M4,0.9,27027,0.898,0.854,Togo,0.015,19.170


  règle de décision : M4 | critère de couverture marginale rempli : True | choix figé dans le code : M4 True


**Constat**

- Dans `regional`, la couverture marginale à 80 % reste comprise entre 78,2 et 80,5 % pour les régions nouvelles et entre 78,6 et 79,7 % pour les années récentes, mais tombe à 72,8 à 74,4 % pour les années récentes de régions nouvelles. Le Soudan n'est couvert qu'à 34,3 à 43,0 % sur les années récentes.
- Dans `survey`, la couverture marginale reste proche de 80 % pour les régions nouvelles. Sur les vagues récentes, elle va de 72,6 % pour M3 à 79,8 % pour M4, et le Mali n'est couvert qu'à 54,3 à 62,9 %.
- Dans `regional`, aucune méthode n'atteint 79 % de couverture marginale dans les trois conditions. La règle de repli du plan retient M0, dont la couverture minimale est la plus élevée, et l'échec du critère est consigné. Dans `survey`, seule M4 remplit le critère, et la règle la retient.

**Analyse**

Le changement spatial seul est bien toléré en moyenne. Le changement temporel, et surtout leur combinaison, dégradent la couverture, d'abord dans quelques pays.

### 3. Écarts à la méthode actuelle

Pour chaque condition, l'écart de couverture marginale à 80 % entre chaque méthode et M0 est accompagné d'un intervalle de confiance à 95 % obtenu par bootstrap des groupes du protocole.

In [4]:
for jeu in JEUX:
    print(jeu)
    display(ecarts_au_m0(jeu, details[jeu], bancs[jeu]).round(4))

regional


,jeu,condition,methode,ecart,ic_bas,ic_haut
0,regional,spatiale,M1,-0.0027,-0.0055,0.0000
1,regional,spatiale,M2,-0.0134,-0.0344,0.0090
2,regional,spatiale,M3,-0.0226,-0.0435,-0.0000
3,regional,spatiale,M4,-0.0218,-0.0428,0.0011
4,regional,temporelle,M1,-0.0039,-0.0080,0.0001
5,regional,temporelle,M2,-0.0061,-0.0171,0.0070
6,regional,temporelle,M3,-0.0087,-0.0204,0.0048
7,regional,temporelle,M4,0.0027,-0.0157,0.0238
8,regional,spatio_temporelle,M1,-0.0067,-0.0118,-0.0019
9,regional,spatio_temporelle,M2,-0.0133,-0.0263,0.0017


survey


,jeu,condition,methode,ecart,ic_bas,ic_haut
0,survey,spatiale,M1,0.0002,-0.0036,0.0041
1,survey,spatiale,M2,0.0023,-0.0016,0.0062
2,survey,spatiale,M3,0.0002,-0.0036,0.0037
3,survey,spatiale,M4,-0.0020,-0.0074,0.0038
4,survey,temporelle,M1,0.0224,0.0193,0.0256
5,survey,temporelle,M2,-0.0313,-0.0362,-0.0264
6,survey,temporelle,M3,-0.0368,-0.0409,-0.0332
7,survey,temporelle,M4,0.0356,0.0310,0.0404
8,survey,spatio_temporelle,M1,0.0202,0.0170,0.0233
9,survey,spatio_temporelle,M2,-0.0317,-0.0362,-0.0272


**Constat**

- Dans `regional`, aucune méthode ne couvre significativement mieux que M0. Pour les années récentes de régions nouvelles, M3 et M1 couvrent significativement moins bien, de 1,6 et 0,7 point. Pour les régions nouvelles, la baisse de 2,3 points de M3 est à la limite de la significativité.
- Dans `survey`, M4 couvre significativement mieux que M0 sur les vagues récentes, de 3,6 points pour les régions connues et de 3,5 points pour les régions nouvelles. M1 couvre aussi significativement mieux, de 2,0 à 2,2 points. M2 et M3 couvrent significativement moins bien, de 3,1 à 3,7 points.

**Analyse**

Stratifier la calibration par pays réduit la couverture moyenne sur les années futures, car les erreurs passées d'un pays ne prévoient pas ses erreurs futures. La normalisation par une difficulté estimée élargit les intervalles là où le modèle se trompe davantage, ce qui profite aux vagues récentes de `survey`.

### 4. Verdict des hypothèses

Les verdicts suivent les énoncés et la règle de réfutation de la section 7 du plan. Une hypothèse est réfutée lorsque le résultat contredit l'énoncé dans l'un des deux jeux. L'énoncé de H2 ne précise pas les conditions concernées, et il est appliqué ici aux trois conditions, ce qui en fait la lecture la plus exigeante.

In [5]:
def verdicts(r):
    a80 = r[r["niveau"] == 0.8].set_index(["condition", "methode"])
    m0 = a80.xs("M0", level="methode")
    h1 = bool((m0["pire_pays"] < 0.75).any())
    h2 = any(all(a80.loc[(c, m), "pire_pays"] >= 0.75
                 and a80.loc[(c, m), "largeur_mediane"] <= 1.25 * m0.loc[c, "largeur_mediane"] for c in CONDITIONS)
             for m in ["M2", "M3"])
    m4 = a80.xs("M4", level="methode")
    h3 = bool((m4["couverture"] >= 0.79).all() and (m4["largeur_mediane"] < m0["largeur_mediane"]).all())
    pertes = {m: {c: a80.loc[("spatiale", m), "couverture"] - a80.loc[(c, m), "couverture"]
                  for c in ["temporelle", "spatio_temporelle"]} for m in METHODES}
    h4 = bool(all(p >= 0.02 for d in pertes.values() for p in d.values()))
    return {"H1": h1, "H2": bool(h2), "H3": h3, "H4": h4}, pertes


tableau = {}
for jeu in JEUX:
    tableau[jeu], pertes = verdicts(resultats[jeu])
    print(jeu, ": perte de couverture marginale à 80 % par rapport à la condition spatiale, en points")
    display((pd.DataFrame(pertes).T * 100).round(1))
verdict = pd.DataFrame(tableau)
verdict["verdict"] = np.where(verdict.all(axis=1), "confirmée", "réfutée")
display(verdict)

regional : perte de couverture marginale à 80 % par rapport à la condition spatiale, en points


,temporelle,spatio_temporelle
M0,1.0,6.1
M1,1.2,6.5
M2,0.3,6.1
M3,-0.3,5.4
M4,-1.4,4.2


survey : perte de couverture marginale à 80 % par rapport à la condition spatiale, en points


,temporelle,spatio_temporelle
M0,3.0,3.0
M1,0.8,1.0
M2,6.4,6.4
M3,6.7,6.6
M4,-0.7,-0.7


,regional,survey,verdict
H1,True,True,confirmée
H2,False,False,réfutée
H3,False,False,réfutée
H4,False,False,réfutée


**Constat**

- **H1 est confirmée.** Avec M0, le pire pays est couvert à 34,3 % dans `regional` et à 57,8 % dans `survey`, pour un niveau nominal de 80 %.
- **H2 est réfutée.** M2 ramène le pire pays de `survey` à 77,0 % pour les régions nouvelles, sans hausse notable de largeur, mais aucune méthode ne le fait sur les années récentes, où le pire pays reste sous 63 % dans `survey` et sous 46 % dans `regional`.
- **H3 est réfutée.** Dans `regional`, M4 ne couvre que 78,3 % des régions nouvelles. Dans `survey`, M4 couvre bien, mais ses intervalles sont plus larges que ceux de M0 sur les vagues récentes, avec un rapport médian de 11,4 contre 9,9.
- **H4 est réfutée.** Dans `regional`, les années récentes de régions connues ne coûtent que 1,2 point au plus, et M3 et M4 y gagnent même. Seule la combinaison avec des régions nouvelles coûte 4,2 à 6,5 points. Dans `survey`, M4 ne perd rien.

**Analyse**

Une seule hypothèse sur quatre est confirmée, et c'est l'hypothèse de diagnostic. Les remèdes envisagés ne corrigent pas l'échec par pays sous changement temporel.

### 5. Évaluation finale sur les ensembles de test

Les cinq méthodes sont calibrées sur toutes les erreurs hors échantillon du développement, puis évaluées une seule fois. Pour M0, les couvertures doivent coïncider avec celles de `12_uncertainty_interpretation.ipynb`, ce qui contrôle la mise en œuvre.

In [6]:
finale = pd.concat([evaluation_finale(jeu) for jeu in JEUX], ignore_index=True)
colonnes = ["jeu", "partition", "methode", "lignes", "couverture", "pire_pays", "pire_pays_nom", "ecart_moyen_pays",
            "largeur_mediane", "pays_evalues"]
for niveau in [0.8, 0.9]:
    print("Niveau nominal", niveau)
    display(finale[finale["niveau"] == niveau][colonnes].round(3))
retenues = finale[(finale["niveau"] == 0.8) & (finale["methode"] == finale["jeu"].map(METHODES_RETENUES))]
m0 = finale[(finale["niveau"] == 0.8) & (finale["methode"] == "M0")].set_index(["jeu", "partition"])
mesures = ["couverture", "pire_pays", "ecart_moyen_pays", "largeur_mediane"]
display(retenues.set_index(["jeu", "partition"])[mesures].join(m0[mesures], rsuffix="_M0").round(3))

Niveau nominal 0.8


,jeu,partition,methode,lignes,couverture,pire_pays,pire_pays_nom,ecart_moyen_pays,largeur_mediane,pays_evalues
0,regional,test_temporel,M0,13221,0.852,0.633,Zimbabwe,0.087,3.023,13
1,regional,test_spatial,M0,22843,0.848,0.719,Niger,0.055,2.964,20
2,regional,test_spatio_temporel,M0,3108,0.845,0.799,Burkina Faso,0.053,3.039,4
6,regional,test_temporel,M1,13221,0.851,0.658,Zimbabwe,0.084,3.200,13
7,regional,test_spatial,M1,22843,0.850,0.726,Burundi,0.052,3.200,20
8,regional,test_spatio_temporel,M1,3108,0.843,0.784,Burkina Faso,0.056,3.200,4
12,regional,test_temporel,M2,13221,0.841,0.727,Mozambique,0.060,3.062,13
13,regional,test_spatial,M2,22843,0.835,0.739,Uganda,0.049,3.085,20
14,regional,test_spatio_temporel,M2,3108,0.801,0.676,Malawi,0.058,2.425,4
18,regional,test_temporel,M3,13221,0.848,0.719,Mozambique,0.062,2.963,13


Niveau nominal 0.9


,jeu,partition,methode,lignes,couverture,pire_pays,pire_pays_nom,ecart_moyen_pays,largeur_mediane,pays_evalues
3,regional,test_temporel,M0,13221,0.928,0.775,Zimbabwe,0.058,4.866,13
4,regional,test_spatial,M0,22843,0.929,0.847,Niger,0.036,4.831,20
5,regional,test_spatio_temporel,M0,3108,0.929,0.893,Burkina Faso,0.040,4.975,4
9,regional,test_temporel,M1,13221,0.926,0.797,Mozambique,0.050,5.377,13
10,regional,test_spatial,M1,22843,0.929,0.825,Burundi,0.036,5.377,20
11,regional,test_spatio_temporel,M1,3108,0.933,0.888,Burkina Faso,0.047,5.377,4
15,regional,test_temporel,M2,13221,0.920,0.822,Mozambique,0.036,5.456,13
16,regional,test_spatial,M2,22843,0.918,0.835,Democratic Republic of the Congo,0.033,5.403,20
17,regional,test_spatio_temporel,M2,3108,0.905,0.814,Malawi,0.045,3.545,4
21,regional,test_temporel,M3,13221,0.923,0.808,Mozambique,0.044,4.448,13


couverture  pire_pays  ecart_moyen_pays  largeur_mediane  couverture_M0  pire_pays_M0  ecart_moyen_pays_M0  largeur_mediane_M0
jeu      partition                                                                                                                                                       
regional test_temporel                          0.852      0.633             0.087            3.023          0.852         0.633                0.087               3.023
         test_spatial                           0.848      0.719             0.055            2.964          0.848         0.719                0.055               2.964
         test_spatio_temporel                   0.845      0.799             0.053            3.039          0.845         0.799                0.053               3.039
survey   test_temporel                          0.758      0.722             0.039            8.873          0.772         0.691                0.029               9.845
         test_spatial                           0.814      0.794             0.026            9.905          0.815         0.768                0.036               9.845
         test_spatio_temporel                   0.753      0.720             0.044            8.897          0.769         0.675                0.032               9.845
         cropcut_grappes_tenues_a_l_ecart       0.809      0.809             0.009            9.282          0.809         0.809                0.009               9.845

**Constat**

- Les couvertures de M0 coïncident avec celles du notebook 12, de 84,5 à 85,2 % pour `regional` et de 76,9 à 81,5 % pour `survey`.
- Dans `regional`, M0 couvre 84,5 à 85,2 % des rendements de test, au-dessus du niveau nominal, mais le Zimbabwe n'est couvert qu'à 63,3 % sur les années futures.
- Dans `survey`, la méthode retenue M4 relève la couverture du pire pays sur chaque ensemble, de 69,1 à 72,2 % sur les vagues futures, de 76,8 à 79,4 % sur les régions nouvelles et de 67,5 à 72,0 % sur leur combinaison. Sa couverture marginale baisse en revanche sur les vagues futures, de 77,2 à 75,8 % et de 76,9 à 75,3 %, alors que le banc d'essai interne annonçait 79,8 %.

**Analyse**

- Le banc d'essai interne a correctement prédit le gain de M4 sur le pire pays, mais pas sa couverture marginale. Les vagues futures du test diffèrent davantage du passé que les vagues récentes du développement. Une validation construite pour imiter un changement de distribution peut donc en sous-estimer l'ampleur.
- Aucune méthode ne garantit la couverture de chaque pays sur les années futures.

### 6. Nature de l'échec par pays

Le Soudan concentre l'échec de `regional` sur le banc d'essai. Ses erreurs et son biais sont comparés entre les années anciennes et récentes.

In [7]:
banc = bancs["regional"]
lignes = []
for role in ["calibration", "temporelle"]:
    soudan = banc[(banc["role"] == role) & (banc["country"] == "Sudan")]
    biais = np.log(soudan["prediction"]) - np.log(soudan["yieldTonHa"])
    lignes.append({"role": role, "lignes": len(soudan), "premiere": int(soudan["harvest_year"].min()),
                   "derniere": int(soudan["harvest_year"].max()), "erreur_mediane": soudan["erreur"].median(),
                   "biais_log_median": biais.median()})
display(pd.DataFrame(lignes).round(3))
temporelle = banc[(banc["role"] == "temporelle") & (banc["country"] == "Sudan")]
print("Biais logarithmique médian par année :",
      (np.log(temporelle["prediction"]) - np.log(temporelle["yieldTonHa"])).groupby(temporelle["harvest_year"]).median().round(2).to_dict())

,role,lignes,premiere,derniere,erreur_mediane,biais_log_median
0,calibration,1166,1981,2011,0.315,0.033
1,temporelle,251,2012,2016,0.766,-0.753


Biais logarithmique médian par année : {2012: -0.21, 2013: -0.61, 2014: -1.03, 2015: -0.7, 2016: -0.98}


**Constat**

- Sur 1 166 lignes de 1981 à 2011, l'erreur médiane vaut 0,315 et le biais logarithmique médian 0,033.
- Sur 251 lignes de 2012 à 2016, l'erreur médiane vaut 0,766 et le biais -0,753. Le biais s'accroît de -0,21 en 2012 à -0,98 en 2016.

**Analyse**

- Le modèle sous-estime d'environ moitié les rendements récents du Soudan. L'échec de couverture provient d'une rupture de niveau, et non d'une dispersion accrue.
- Un intervalle centré sur une prédiction biaisée ne peut pas couvrir, quelle que soit sa largeur raisonnable. Les méthodes conformes, qui ajustent la largeur, sont donc impuissantes face à ce type de changement.
- La rupture commence l'année qui suit la sécession du Soudan du Sud, en 2011. Ce rapprochement reste une hypothèse, que les données de l'étude ne permettent pas d'établir.

### 7. Synthèse

**Résultats**

| Élément | `regional` | `survey` |
|---|---|---|
| Méthode retenue par la règle | M0, critère non rempli | M4, critère rempli |
| Couverture marginale de test à 80 % | 84,5 à 85,2 % | 75,3 à 81,4 % |
| Pire pays sur les années futures de test | 63,3 % | 72,0 à 72,2 % |
| Hypothèses | H1 confirmée, H2, H3 et H4 réfutées | Idem |

**Enseignements**

- Une incertitude calibrée en moyenne peut être gravement trop confiante pour un sous-groupe identifiable, sans que la métrique globale le révèle.
- Sous changement temporel, l'échec par groupe vient surtout d'un biais du modèle. Élargir les intervalles ne suffit pas, il faut détecter le changement.
- Un banc d'essai qui imite le changement de distribution aide à choisir, mais peut en sous-estimer l'ampleur.

**Limites**

- Le problème a été repéré sur les ensembles de test, avant l'étude.
- Le banc d'essai de `survey` ne compte que six pays d'au moins 200 lignes récentes, et le test spatio-temporel de `regional` quatre pays.
- Le bootstrap des groupes ignore la corrélation entre groupes voisins et entre années d'un même pays.

**Suite proposée**

Les rendements arrivent une fois par an. L'inférence conforme adaptative, proposée par Gibbs et Candès en 2021, réajuste le niveau des intervalles à mesure que les erreurs récentes sont observées. Une seconde étude préenregistrée pourrait tester si elle rétablit la couverture des pays en rupture, comme le Soudan, en une ou deux saisons.